# Tetrahedral fields from a four-ray linear-gradient trace

This example launches a $2\times2$ parallel ray lattice at 20 degrees into the same linear density gradient as the turning-point regression. The two samples along the beam's second spot axis lie on opposite sides of $z=0$, giving each sheet finite out-of-plane thickness. Ten path samples produce $6(2-1)(2-1)(10-1)=54$ tetrahedra per sheet before degeneracy checks.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pyGATH.fields import (
    interpolate_tetrahedral_fields_batched,
    tetrahedralise_sheet_fields,
)
from pyGATH.io import load_simulation_config
from pyGATH.plotting import plot_tetrahedral_mesh
from pyGATH.raytracing import RAY_STATE_LAYOUT

In [ ]:
repo_root = Path.cwd()
if not (repo_root / "configs").is_dir():
    repo_root = repo_root.parent

deck = repo_root / "configs" / "example_configs" / "tetrahedral_linear_gradient.toml"
simulation = load_simulation_config(deck)
grid = simulation.build_grid()
initial_rays = simulation.initialize_rays(grid)
initial_positions = np.asarray(initial_rays.state[0, ..., RAY_STATE_LAYOUT.position])
initial_z = initial_positions[..., 2]
initial_face_span_yz = np.ptp(initial_positions[..., 1:], axis=(0, 1))
x_min = float(grid.extents[0, 0])
assert initial_z.min() < 0.0 < initial_z.max()
np.testing.assert_allclose(initial_face_span_yz, 100.0e-6, atol=1.0e-12)
np.testing.assert_allclose(initial_positions[..., 0], x_min, atol=2.0e-9)

result = simulation.trace_rays(initial_rays, grid)
assert result.sheet_fields.shape == (1, 2, 2, 2, 10, 43)
assert np.all(np.asarray(result.has_caustic))
caustic_xy = np.asarray(result.sheet_fields[0, 0, ..., -1, :2])
caustic_x_spread = np.ptp(caustic_xy[..., 0])
assert caustic_x_spread < 1.0e-9

print(f"Initial ray z coordinates [m]: {np.unique(initial_z)}")
print(f"Initial face y-z dimensions [m]: {initial_face_span_yz}")
print(f"Initial ray x coordinates [m]: {np.unique(initial_positions[..., 0])}")
print(f"Sheet field shape: {result.sheet_fields.shape}")
print(f"Caustic mask:\n{np.asarray(result.has_caustic[0])}")
print(f"Caustic x-y coordinates [m]:\n{caustic_xy.reshape(-1, 2)}")
print(f"Caustic x spread [m]: {caustic_x_spread:.3e}")

In [ ]:
tetrahedral_field = tetrahedralise_sheet_fields(
    result.sheet_fields,
    fields=("phase_length", "path_length"),
)
expected_tetrahedra = 6 * (2 - 1) * (2 - 1) * (10 - 1)
valid = np.asarray(tetrahedral_field.mesh.valid[0])
valid_counts = valid.sum(axis=-1)
assert tetrahedral_field.mesh.ntetrahedra == expected_tetrahedra
assert np.all(valid_counts >= 5)

print(f"Tetrahedra per sheet: {expected_tetrahedra}")
print(f"Valid tetrahedra in sheets 1 and 2: {valid_counts}")

In [ ]:
random_generator = np.random.default_rng(1424)
highlighted_tetrahedra = []
for sheet_index in range(2):
    valid_indices = np.flatnonzero(valid[sheet_index])
    highlighted_tetrahedra.append(
        tuple(
            int(index)
            for index in random_generator.choice(valid_indices, size=5, replace=False)
        )
    )
highlighted_tetrahedra

In [ ]:
edge_colors = ("lime", "cyan", "deeppink", "orange", "yellow")
vertex_colors = ("green", "darkcyan", "mediumvioletred", "darkorange", "olive")

for sheet_index in range(2):
    figure = plt.figure(figsize=(10, 7))
    axis = figure.add_subplot(111, projection="3d")
    plot_tetrahedral_mesh(
        tetrahedral_field,
        beam_index=0,
        sheet_index=sheet_index,
        highlighted_tetrahedra=highlighted_tetrahedra[sheet_index],
        highlight_edge_colors=edge_colors,
        highlight_vertex_colors=vertex_colors,
        ax=axis,
    )
    axis.set_title(
        f"Sheet {sheet_index + 1}: all tetrahedra and five highlighted examples"
    )
    # Expand the visually thin z direction so individual tetrahedra can be inspected.
    # axis.set_aspect('equal')#(1.0, 1.0, 1.0))
    # axis.set_zlim(-2.0e-4, 2.0e-4)
    axis.grid(False)
    axis.view_init(elev=52, azim=-0)
    figure.tight_layout()
    plt.show()

## Interpolation on the $z=0$ plane

The query grid covers the xy footprint of both sheets. Values outside each individual sheet remain exactly zero in the interpolation result, but are masked in the plots so the narrow curved sheet footprint is visible.

In [ ]:
vertices = np.asarray(tetrahedral_field.mesh.vertex_positions[0])
x_limits = (vertices[..., 0].min(), vertices[..., 0].max())
y_limits = (vertices[..., 1].min(), vertices[..., 1].max())
x_padding = 0.02 * (x_limits[1] - x_limits[0])
y_padding = 0.02 * (y_limits[1] - y_limits[0])
x = np.linspace(x_limits[0] - x_padding, x_limits[1] + x_padding, 260)
y = np.linspace(y_limits[0] - y_padding, y_limits[1] + y_padding, 360)
x_mesh, y_mesh = np.meshgrid(x, y, indexing="xy")
query_points = np.column_stack((x_mesh.ravel(), y_mesh.ravel(), np.zeros(x_mesh.size)))
interpolated = interpolate_tetrahedral_fields_batched(
    tetrahedral_field, query_points, point_batch_size=8192
)
values = np.asarray(interpolated.values[0])
inside = np.asarray(interpolated.inside[0])

assert np.all(values[~inside] == 0.0)
assert np.all(inside.sum(axis=-1) > 0)
phase = values[..., tetrahedral_field.selection.phase_length]
path = values[..., tetrahedral_field.selection.path_length]
assert np.all(phase[inside] >= 0.0)
assert np.all(phase[inside] <= path[inside] + 1e-12)
print(f"Inside query points in sheets 1 and 2: {inside.sum(axis=-1)}")

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(13, 10), sharex=True, sharey=True)
plot_fields = (
    ("Phase length", tetrahedral_field.selection.phase_length),
    ("Path length", tetrahedral_field.selection.path_length),
)

for sheet_index in range(2):
    sheet_vertices = vertices[sheet_index]
    for field_index, (field_name, compact_index) in enumerate(plot_fields):
        axis = axes[sheet_index, field_index]
        plane_values = values[sheet_index, :, compact_index].reshape(x_mesh.shape)
        plane_inside = inside[sheet_index].reshape(x_mesh.shape)
        masked_values = np.ma.masked_where(~plane_inside, plane_values)
        image = axis.pcolormesh(
            x_mesh, y_mesh, masked_values, shading="auto", cmap="viridis"
        )
        axis.scatter(
            sheet_vertices[:, 0],
            sheet_vertices[:, 1],
            color="black",
            s=3,
            alpha=0.35,
        )
        axis.set_title(f"Sheet {sheet_index + 1}: {field_name}")
        axis.set_xlabel("x [m]")
        axis.set_ylabel("y [m]")
        axis.set_aspect("equal", adjustable="box")
        figure.colorbar(image, ax=axis, label=f"{field_name} [m]")

figure.suptitle("Barycentric interpolation on the z=0 plane")
figure.tight_layout()
plt.show()